# 📝 RAG 파이프라인 과제 LV3 정답 — 재고·진단하고·고쳐·다시 재기 (강사용)

단계별 **모범답안 + 해설**입니다. 학생이 스스로 푼 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day16_RAG_파이프라인/data/` 이고, `.env` 는 `../../day16_RAG_파이프라인/.env` 를 함께 읽습니다.
- 지표 값은 모두 실측입니다. 자가채점은 정확일치 대신 **범위와 방향**으로 검사합니다.
- 마지막 문제는 서술형입니다 — 아래 모범 서술과 학생 답을 비교해 주세요.

---
## 준비 — 지난 강의에서 만든 도구 되살리기
아래 셀들은 **실행만** 하세요.

`search_docs(col, query, k)` 는 조각 검색 결과를 **문서 단위로 접어** 서로 다른 문서 `k` 개를 돌려줍니다(LV2 에서 직접 만들어 본 그 함수예요). 이번에는 도구로 제공하니 **평가와 개선에 집중**하세요.

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 이 셀은 실행만 하세요.
# 15일차와 같은 방식입니다: .env 의 OPENAI_API_KEY 로 실제 OpenAI 에 연결합니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../../day16_RAG_파이프라인/.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인한다 — OpenAI() 를 만든 뒤에 검사하면 SDK 인증 오류가 먼저 나서 이 안내가 묻힌다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from openai import OpenAI

client = OpenAI()
print("OpenAI 클라이언트 준비 완료 — 실제 API 연결됨")

In [ ]:
# [제공 코드] 문서와 평가셋을 불러옵니다
import pandas as pd

qna_docs = pd.read_csv('../../day16_RAG_파이프라인/data/qna_docs.csv')
qna_eval = pd.read_csv('../../day16_RAG_파이프라인/data/qna_eval.csv')
print(f'문서 수: {len(qna_docs)}   평가 질문 수: {len(qna_eval)}')
display(qna_eval.head(3))

In [ ]:
# [제공 코드] 임베딩 모델 준비 — 지난 시간에 쓴 한국어 문장 임베딩 모델입니다(불러오는 데 잠시 걸립니다).
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 (768차원)')

In [ ]:
# [제공 코드] 청킹 함수 — 이 강의 앞부분에서 만든 세 가지 청킹 전략입니다.
def chunk_fixed(text, size):
    """고정 크기(글자 수)로 자른다."""
    return [text[i:i + size] for i in range(0, len(text), size)]

def chunk_overlap(text, size, overlap):
    """앞 청크의 끝 일부를 다음 청크가 겹쳐 갖도록 자른다."""
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)]

def chunk_paragraph(text, size):
    """빈 줄로 나뉜 문단을 순서대로 모아 size 근처에서 끊는다.

    한 문단 자체는 절대 쪼개지 않는다 -- 문단이 size 보다 길면 그 조각 하나가
    size 를 그대로 넘어선 채로 들어간다."""
    chunks, cur = [], ''
    for para in [p.strip() for p in text.split('\n\n') if p.strip()]:
        if cur and len(cur) + len(para) > size:
            chunks.append(cur)
            cur = para
        else:
            cur = f'{cur}\n{para}' if cur else para
    if cur:
        chunks.append(cur)
    return chunks

In [ ]:
# [제공 코드] 색인·검색 도구 — 지난 시간(임베딩·벡터DB)에 배운 것을 함수로 묶어 둡니다.
import hashlib
from pathlib import Path

import chromadb

# 지난 시간에는 EphemeralClient(메모리)를 썼습니다. 오늘 문서는 수십 쪽이라 임베딩에 시간이 걸리니,
# PersistentClient 로 **디스크에 저장**합니다. 한 번 만들어 두면 커널을 새로 켜도 그대로 남아 있어
# 다시 임베딩하지 않습니다. (output/ 폴더는 실행 산출물이라 저장소에 올라가지 않습니다.)
CHROMA_DIR = Path('output' if Path('data').exists() else 'output') / 'chroma'
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

def index_fingerprint(ids, texts, metadatas):
    """색인에 담긴 내용을 한 줄로 요약한 지문. 무엇 하나라도 바뀌면 값이 달라진다.

    메타데이터까지 넣는 이유: 본문이 그대로여도 메타에 열이 하나 늘면(예: 쪽 번호) 낡은
    색인에는 그 열이 없다. 그걸 모르고 다시 쓰면 검색은 되는데 meta['article'] 에서 KeyError 가 난다.
    """
    parts = ['\n'.join(ids), '\n'.join(texts),
             '\n'.join(repr(sorted(m.items())) for m in metadatas)]
    return hashlib.sha256('\x00'.join(parts).encode()).hexdigest()[:16]

def make_index(ids, texts, metadatas, name):
    """청크를 임베딩해 컬렉션으로 만든다. 같은 내용으로 이미 만들어 뒀으면 그대로 다시 쓴다."""
    want = index_fingerprint(ids, texts, metadatas)

    got = chroma.get_or_create_collection(name, metadata={'hnsw:space': 'cosine', 'fp': want})
    if got.count() == len(ids) and (got.metadata or {}).get('fp') == want:
        print(f'{name}: 만들어 둔 색인을 그대로 씁니다 (청크 {got.count()}개)')
        return got

    # 개수나 지문이 다르면 문서가 바뀐 것이다 — 낡은 색인을 지우고 새로 만든다.
    chroma.delete_collection(name)
    col = chroma.create_collection(name, metadata={'hnsw:space': 'cosine', 'fp': want})

    emb = embed_model.encode(texts, normalize_embeddings=True)
    col.add(ids=ids, embeddings=emb.tolist(), documents=texts, metadatas=metadatas)

    print(f'{name}: 색인을 새로 만들었습니다 (청크 {col.count()}개)')
    return col

def search(col, query, k):
    """질문과 가장 가까운 청크 k개의 본문을 돌려준다."""
    qe = embed_model.encode([query], normalize_embeddings=True)
    res = col.query(query_embeddings=qe.tolist(), n_results=k)
    return res['documents'][0]

def search_docs(col, query, k):
    """상위 청크의 부모 문서 id 를 중복 없이 앞에서부터 k개 돌려준다(지표 계산용)."""
    qe = embed_model.encode([query], normalize_embeddings=True)
    res = col.query(query_embeddings=qe.tolist(), n_results=k * 5)
    seen = []
    for m in res['metadatas'][0]:
        if m['doc_id'] not in seen:
            seen.append(m['doc_id'])
        if len(seen) >= k:
            break
    return seen

In [ ]:
# [제공 코드] 검색 품질 지표 — 지난 강의(평가)에서 손으로 구현한 네 가지 지표입니다.
def hit_at_k(predicted, relevant, k):
    """상위 k개 중 관련 문서가 하나라도 있으면 1, 없으면 0."""
    return 1 if any(p in relevant for p in predicted[:k]) else 0

def precision_at_k(predicted, relevant, k):
    """상위 k개 중 관련 문서의 비율(관련 수 / k)."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / k

def recall_at_k(predicted, relevant, k):
    """전체 관련 문서 중 상위 k개가 찾아낸 비율(관련 수 / 전체 관련 수)."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / len(relevant)

def mrr(predicted, relevant):
    """첫 번째 관련 문서 순위의 역수(1위면 1, 2위면 1/2 …). 없으면 0."""
    for i, p in enumerate(predicted, 1):
        if p in relevant:
            return 1 / i
    return 0.0

---
## 1단계. 기준이 될 파이프라인 만들기
무엇을 고칠지 정하려면 **먼저 기준선**이 있어야 합니다. 가장 단순한 방식 — 질의응답 한 건을 **통째로** 색인하는 것 — 부터 만듭니다. 문서 93건이 그대로 93개의 색인 항목이 됩니다.

**이 단계에서 만들 것**:
- 문서 id 리스트 **`base_ids`** — `qna_docs['id']`
- 문서 본문 리스트 **`base_texts`** — `qna_docs['본문']`
- 메타데이터 리스트 **`base_metas`** — 키는 **`'doc_id'`·`'field'`(분야)·`'article'`(조항)·`'page'`(쪽, 정수)**
- `make_index(...)` 로 컬렉션 이름 **`'qna_lv3_base'`** 의 색인을 만들어 **`base_col`** 에 담기

**확인**: `base_col.count()` 가 **93** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 자르지 않으므로 표의 열을 그대로 리스트로 바꾸면 된다. 메타데이터만 행 단위로 만든다.

세부구현:
1. id 열과 본문 열을 각각 리스트로 만든다.
2. 표를 행 단위로 돌며 네 키를 가진 딕셔너리를 만들어 리스트에 모은다(쪽은 정수로).
3. 세 리스트와 컬렉션 이름을 색인 함수에 넘겨 결과를 변수에 담는다.
```

</details>

In [ ]:
base_ids = qna_docs['id'].tolist()
base_texts = qna_docs['본문'].tolist()
base_metas = [{'doc_id': r['id'], 'field': r['분야'],
               'article': r['조항'], 'page': int(r['쪽'])}
              for _, r in qna_docs.iterrows()]

base_col = make_index(base_ids, base_texts, base_metas, 'qna_lv3_base')
print(f'기준 색인: {base_col.count()}건')

In [ ]:
# [자가채점]
assert base_col.count() == 93
assert base_texts[0] in qna_docs['본문'].tolist(), '본문을 그대로 넣어야 합니다'
assert set(base_metas[0]) == {'doc_id', 'field', 'article', 'page'}
assert all(isinstance(mm['page'], int) for mm in base_metas)
# 검색이 되는지 한 번 확인한다
assert search_docs(base_col, '탈퇴한 회원의 회원번호만 남겨 두어도 되나요', 1) == ['q64']
print('✅ 통과!')

### 해설

**기준선(baseline)을 먼저 만든다**가 이 단계의 전부입니다. 개선은 언제나 "무엇에 견주어" 라는 말을 필요로 하는데, 기준을 안 만들어 두면 나중에 바꾼 결과가 좋아진 건지 알 수 없습니다.

메타데이터에 분야·조항·쪽을 실어 두는 것도 습관으로 삼으세요. 지금 문제에서는 안 쓰지만 마무리 문제에서 출처를 붙일 때 필요합니다.

---
## 2단계. 평가셋으로 재기
`qna_eval` 의 질문 15개를 모두 검색해 네 지표의 평균을 냅니다. **K=3** 으로 잽니다.

**이 단계에서 만들 것**:
- 각 행의 `query` 를 `search_docs(base_col, query, 3)` 으로 검색합니다.
- 정답은 `relevant` 열의 문자열입니다. 정답이 여럿이면 **`|` 로 이어져** 있으니 `str(...).split('|')` 로 리스트로 만드세요.
- 네 지표의 평균을 딕셔너리 **`base_scores`** 에 담으세요. 키는 **`'hit'`·`'precision'`·`'recall'`·`'mrr'`** 입니다.
- **MRR 도 상위 3개 목록 안에서** 잽니다 — 검색해 온 그 목록을 그대로 넘기세요(더 깊이까지 뽑아 재면 값이 달라집니다).
- 질문별 결과도 한 줄씩 출력하세요 — **3단계에서 실패를 찾으려면 눈으로 봐야 합니다.**

**확인**: `base_scores['hit']` 는 **1보다 작습니다**(모두 맞히지는 못합니다).

<details><summary>힌트</summary>

```text
접근방법:
- 지표별 리스트에 값을 모아 두고 마지막에 평균을 낸다. MRR 만 K 를 받지 않는다.

세부구현:
1. 지표별로 빈 리스트를 만든다.
2. 평가셋을 행 단위로 돌며 질문과 정답 문자열을 꺼낸다.
   2-1. 정답 문자열을 구분자로 나눠 리스트로 만든다.
   2-2. 기준 색인에서 문서 3개를 검색한다.
   2-3. 네 지표를 각각 계산해 리스트에 넣고, 결과를 한 줄 출력한다.
3. 각 리스트의 합을 개수로 나눠 딕셔너리에 담는다.
```

</details>

In [ ]:
def measure(col):
    hits, precisions, recalls, rrs = [], [], [], []
    for _, row in qna_eval.iterrows():
        relevant = str(row['relevant']).split('|')
        predicted = search_docs(col, row['query'], 3)
        hits.append(hit_at_k(predicted, relevant, 3))
        precisions.append(precision_at_k(predicted, relevant, 3))
        recalls.append(recall_at_k(predicted, relevant, 3))
        rrs.append(mrr(predicted, relevant))
        print(f"{hits[-1]} {predicted} <- {relevant} | {row['query'][:26]}")
    n = len(hits)
    return {'hit': sum(hits) / n, 'precision': sum(precisions) / n,
            'recall': sum(recalls) / n, 'mrr': sum(rrs) / n}

base_scores = measure(base_col)
print()
for name, value in base_scores.items():
    print(f'{name}: {value:.3f}')

In [ ]:
# [자가채점]
assert set(base_scores) == {'hit', 'precision', 'recall', 'mrr'}
# 손으로 적은 값이 아니라 실제로 잰 값인지 — 네 지표를 모두 다시 계산해 대조한다
check = {'hit': [], 'precision': [], 'recall': [], 'mrr': []}
for _, row in qna_eval.iterrows():
    relevant = str(row['relevant']).split('|')
    predicted = search_docs(base_col, row['query'], 3)
    check['hit'].append(hit_at_k(predicted, relevant, 3))
    check['precision'].append(precision_at_k(predicted, relevant, 3))
    check['recall'].append(recall_at_k(predicted, relevant, 3))
    check['mrr'].append(mrr(predicted, relevant))
for name, values in check.items():
    assert abs(base_scores[name] - sum(values) / len(values)) < 1e-9, \
        f'{name} 평균이 실제 검색 결과와 다릅니다'
# 실측 기준선: Hit@3 0.867 · MRR 0.833
assert 0.80 < base_scores['hit'] < 0.95, f"Hit@3 이 {base_scores['hit']:.3f} 입니다"
assert 0.78 < base_scores['mrr'] < 0.90
assert base_scores['mrr'] <= base_scores['hit'], 'MRR 은 Hit 보다 클 수 없습니다'
# Precision@3 의 상한은 문항마다 다르다 — 정답이 1개면 1/3, 2개면 2/3 이 최대다
sizes = [len(str(r).split('|')) for r in qna_eval['relevant']]
ceiling = sum(min(n, 3) for n in sizes) / (3 * len(sizes))
assert base_scores['precision'] < ceiling, \
    f'Precision@3 이 상한 {ceiling:.3f} 을 넘었습니다 — 계산을 다시 보세요'
assert base_scores['recall'] < 1.0
print('✅ 통과!')

### 해설

실측 기준선은 `hit=0.867`, `precision=0.356`, `recall=0.833`, `mrr=0.833` 입니다.

**네 가지를 읽어야 합니다.** 1) Hit 이 0.867 이라는 것은 15문항 중 **2문항에서 정답을 상위 3에 못 넣었다**는 뜻입니다. 2) MRR(0.833)이 Hit(0.867)보다 낮은 것은 맞힌 질문 중 **1위가 아닌 것이 섞여 있다**는 뜻이에요. 3) Recall(0.833)이 Hit 보다 **더** 낮은 것은, **정답이 둘인 문항에서 하나만 찾은** 경우가 있다는 뜻입니다 — Hit 은 하나만 맞혀도 1로 세지만 Recall 은 절반으로 셉니다. 4) Precision 0.356 은 낮아 보이지만, 정답이 1개인 문항을 3개 가져와 재면 1/3 이 최대라 **이 평가셋의 상한 자체가 0.422**입니다.

> 평가셋 15문항 중 **4문항은 정답이 둘**입니다(`q14|q22`·`q42|q47`·`q70|q71`·`q86|q91`). 그래서 네 지표가 서로 다른 것을 재게 됩니다 — 정답이 전부 하나였다면 Recall 은 Hit 과 **같은 값**이 되어 볼 이유가 없었을 거예요.

평균만 보고 끝내면 여기서 멈춥니다. 다음 단계에서 **어느 질문이 왜 실패했는지**를 봅니다.

---
## 3단계. 실패를 진단하기
평균은 "어딘가 문제가 있다"까지만 말해 줍니다. 고치려면 **어느 질문이 실패했고 그 정답 문서가 어떤 문서인지**를 봐야 합니다.

**이 단계에서 만들 것**:
- 기준 색인에서 **Hit@3 이 0** 인 질문들을 찾아, 그 행의 **정답 문서 id** 를 모아 리스트 **`failed_docs`** 에 담으세요(평가셋에 나온 순서대로, `|` 가 있으면 나눠서 모두).
- **같은 반복 안에서**, 그 질문이 실제로 찾아온 **상위 3 목록**을 리스트 **`failed_predicted`** 에 담으세요(실패한 질문 하나당 목록 하나, 같은 순서로).
- 그 문서들이 어떤 문서인지 표로 확인하세요 — **`본문` 의 글자 수**를 함께 보세요.
- 문서 전체의 평균 글자 수도 함께 출력해 비교하세요.

**확인**: 실패한 정답 문서는 **2건**이고, 둘 다 본문이 전체 평균보다 **깁니다**. `failed_predicted` 는 길이 3 인 목록 **2개**를 담은 리스트입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2단계와 같은 방식으로 검색하되, 이번에는 값이 아니라 '어느 행이 0이었나'를 남긴다.

세부구현:
1. 빈 리스트 두 개를 만들고 평가셋을 행 단위로 돈다.
2. 정답 문자열을 나눠 리스트로 만들고, 기준 색인에서 문서 3개를 검색한다.
3. Hit 지표가 0이면 그 행의 정답 id 들을 첫 리스트에 더하고,
   3-1. 방금 검색해 온 상위 3 목록을 둘째 리스트에 그대로 덧붙인다.
4. 문서 표에서 그 id 들의 행만 골라, 본문 글자 수를 새 열로 만들어 함께 본다.
5. 전체 본문 글자 수의 평균도 출력해 견준다.
```

</details>

In [ ]:
failed_docs, failed_predicted = [], []
for _, row in qna_eval.iterrows():
    relevant = str(row['relevant']).split('|')
    predicted = search_docs(base_col, row['query'], 3)
    if hit_at_k(predicted, relevant, 3) == 0:
        failed_docs += relevant
        failed_predicted.append(predicted)
        print(f"놓침: {relevant} <- {row['query']}")
        print(f'         찾아온 것: {predicted}')

lengths = qna_docs.assign(글자수=qna_docs['본문'].str.len())
display(lengths[lengths['id'].isin(failed_docs)][['id', '분야', '조항', '쪽', '글자수', '질문']])
print(f"전체 평균 글자 수: {lengths['글자수'].mean():.0f}")

In [ ]:
# [자가채점]
assert failed_docs == ['q30', 'q90'], f'놓친 문서 목록이 다릅니다: {failed_docs}'
# 진단 산출물이 실제 검색 결과인지 대조한다 — 목록을 손으로 적어서는 맞출 수 없다
assert len(failed_predicted) == len(failed_docs), \
    'failed_predicted 는 실패한 질문 하나당 목록 하나여야 합니다'
assert all(len(p) == 3 for p in failed_predicted)
for doc_id, predicted in zip(failed_docs, failed_predicted):
    hit_row = qna_eval[qna_eval['relevant'].apply(
        lambda r: doc_id in str(r).split('|'))].iloc[0]
    assert predicted == search_docs(base_col, hit_row['query'], 3), \
        f'{doc_id} 질문의 상위 3 목록이 실제 검색 결과와 다릅니다 — 직접 검색한 목록을 담으세요'
    assert doc_id not in predicted, f'{doc_id} 는 실패한 질문이 아닙니다'
# 놓친 문서는 둘 다 평균보다 길다 — 이게 다음 단계의 실마리다
avg_len = qna_docs['본문'].str.len().mean()
assert all(len(qna_docs.set_index('id').loc[d, '본문']) > avg_len for d in failed_docs)
print('✅ 통과!')

### 해설

놓친 두 문서는 **`q30`(만족도 조사, 770자)** 과 **`q90`(클라우드 수탁자, 1,043자)** 입니다. 전체 평균이 629자이니 둘 다 평균보다 길고, `q90` 은 93건 중 **두 번째로 긴** 문서입니다.

왜 긴 문서가 불리할까요? 문서를 **통째로 임베딩하면 벡터 하나가 그 문서 전체를 뭉뚱그려** 나타냅니다. `q90` 은 클라우드 사업자의 지위·계약·감독·책임을 두루 다루는데, 질문은 그중 "계약을 따로 맺어야 하나" 한 대목만 묻습니다. 문서 전체의 평균적인 뜻과 질문의 뜻이 어긋나 순위가 밀린 것입니다.

여기까지 오면 고칠 곳이 보입니다 — **문서를 더 작게 나누면** 그 한 대목이 자기 벡터를 갖게 됩니다. 다음 단계에서 청킹만 바꿔 봅니다.

---
## 4단계. 청킹만 바꿔 다시 색인하기
진단이 가리키는 곳은 **문서를 통째로 넣은 것**이었습니다. 그러니 **청킹**만 바꿉니다. 임베딩 모델·검색 방식·K 는 그대로 둡니다 — **한 번에 하나만 바꿔야** 좋아진 이유를 알 수 있습니다.

**이 단계에서 만들 것**:
- 각 문서의 `본문` 을 **`chunk_paragraph(본문, 300)`** 으로 자릅니다.
- 조각 id 는 **`f'{문서 id}-{조각 번호}'`**(문서 안에서 **0부터**), 메타데이터 키는 1단계와 **같게** 네 개입니다.
- `make_index(...)` 로 컬렉션 이름 **`'qna_lv3_tuned'`** 의 색인을 만들어 **`tuned_col`** 에 담으세요.
- 2단계와 **똑같은 방법**으로 재서 딕셔너리 **`tuned_scores`** 에 담으세요(키도 같게).

**확인**: 조각은 **282개**가 나오고, `tuned_scores['hit']` 는 **1.0** 입니다.

> 재는 코드를 두 번 쓰지 마세요. 2단계에서 **함수로 만들어 뒀다면** 색인만 바꿔 다시 부르면 됩니다. 같은 방식으로 재야 비교가 성립합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 자르는 부분만 새로 쓰고, 재는 부분은 2단계에서 쓴 것을 그대로 다시 쓴다.

세부구현:
1. 빈 리스트 세 개를 만들고 문서를 행 단위로 돈다.
2. 본문을 문단 청킹하고, 조각을 번호와 함께 돌며 id·본문·메타데이터를 채운다.
3. 색인 함수에 새 컬렉션 이름으로 넘겨 결과를 변수에 담는다.
4. 2단계와 같은 방식으로 평가셋 전체를 재어 딕셔너리에 담는다.
```

</details>

In [ ]:
tuned_ids, tuned_texts, tuned_metas = [], [], []
for _, row in qna_docs.iterrows():
    for i, piece in enumerate(chunk_paragraph(row['본문'], 300)):
        tuned_ids.append(f"{row['id']}-{i}")
        tuned_texts.append(piece)
        tuned_metas.append({'doc_id': row['id'], 'field': row['분야'],
                            'article': row['조항'], 'page': int(row['쪽'])})

tuned_col = make_index(tuned_ids, tuned_texts, tuned_metas, 'qna_lv3_tuned')
print(f'조각 수: {len(tuned_ids)}')
print()
tuned_scores = measure(tuned_col)
print()
for name, value in tuned_scores.items():
    print(f'{name}: {base_scores[name]:.3f} -> {value:.3f}')

In [ ]:
# [자가채점]
assert tuned_col.count() == 282, \
    f'조각이 {tuned_col.count()}개입니다 — chunk_paragraph 에 300 을 넘겼는지 확인하세요'
assert set(tuned_scores) == set(base_scores)
# 실제로 잰 값인지 다시 계산해 대조한다
recheck = []
for _, row in qna_eval.iterrows():
    predicted = search_docs(tuned_col, row['query'], 3)
    recheck.append(hit_at_k(predicted, str(row['relevant']).split('|'), 3))
assert abs(tuned_scores['hit'] - sum(recheck) / len(recheck)) < 1e-9
# 청킹을 바꾼 뒤 실제로 좋아졌는지 — 방향으로 검사한다
assert tuned_scores['hit'] == 1.0, '15문항을 모두 상위 3에 넣지 못했습니다'
assert tuned_scores['mrr'] > base_scores['mrr']
assert tuned_scores['recall'] > base_scores['recall']
assert tuned_scores['precision'] > base_scores['precision']
# 기준 색인이 덮어써지지 않았는지 — 두 색인이 함께 살아 있어야 비교가 성립한다
assert base_col.count() == 93 and tuned_col.count() > base_col.count()
print('✅ 통과!')

### 해설

`93`건이 `282`조각이 되면서 지표가 이렇게 움직였습니다.

| 지표 | 기준(통째) | 개선(문단 300자) | 이 평가셋의 최대 |
|---|---|---|---|
| Hit@3 | 0.867 | **1.000** | 1.000 |
| Precision@3 | 0.356 | 0.400 | 0.422 |
| Recall@3 | 0.833 | 0.967 | 1.000 |
| MRR | 0.833 | 0.922 | 1.000 |

### 이 화면을 '다 풀었다'로 읽으면 안 됩니다

**Hit@3 만 천장에 닿았습니다.** 나머지 셋은 아직 갈 길이 남아 있고, 각각 다른 것을 말합니다.

**1) Precision@3 0.400 은 상한이 1.000 이 아닙니다.** 이 평가셋에서 정답은 문항당 1~2개인데 상위 3칸을 가져오므로, 아무리 완벽해도 **0.422 가 최대**입니다. 0.400 은 거의 그 끝이에요. 0.4 를 보고 "정밀도가 낮다"고 오해하면 엉뚱한 곳을 고치게 됩니다. **지표는 값이 아니라 그 값이 닿을 수 있는 최대와 견줘 읽습니다.**

**2) Recall@3 0.967 — 여기에 아직 실패가 하나 숨어 있습니다.** "동의를 안 하면 가입이나 결제가 안 되게 막아도 되나요" 문항은 정답이 `q70`·`q71` 둘인데 `q70` 만 찾았습니다. **Hit 은 이걸 1 로 세지만 Recall 은 0.5 로 셉니다** — Hit 만 봤다면 못 봤을 실패예요.

**3) MRR 0.922 — `q30` 이 아직 3위입니다.** 마무리 1에서 직접 확인하겠지만, `q30`(만족도 조사)은 상위 3에 **겨우 턱걸이**한 것이지 잘 찾은 게 아닙니다. 상위 3은 `q48`·`q33`·`q30` 순이라 **엉뚱한 두 건이 앞을 막고 있습니다.** Hit@3 은 1 로 세지만 MRR 은 0.333 으로 셉니다.

**Hit@3 이 1.000 이라는 것은 "완벽해졌다"가 아니라 "이 평가셋으로는 더 잴 수 없다"는 뜻입니다.** 개선을 한 번 더 한다면 **Recall(놓친 둘째 정답)과 MRR(순위)** 을 보고, 그래도 부족하면 K 를 1 로 낮춰 더 엄하게 재면 됩니다.

Precision 이 0.289 → 0.333 으로 오른 것도 같은 이야기입니다 — **상한이 1/3** 이므로 0.333 은 "모든 질문에서 정답을 상위 3 안에 정확히 하나 넣었다", 즉 **상한에 닿았다**는 뜻입니다.

**한 번에 하나만 바꾼 것**이 이 단계의 핵심입니다. 청킹과 함께 모델이나 K 까지 바꿨다면 무엇이 효과를 냈는지 알 수 없고, 다음에 비슷한 문제를 만났을 때 써먹을 지식이 남지 않습니다.

참고로 다른 후보도 재 봤습니다 — 고정 200자 MRR 0.933, 고정 300자 0.889, 오버랩 300/100 0.922, 문단 400자 0.900. **어느 것이 최고인지는 자료마다 다르므로 재 보는 수밖에 없습니다.** 문단 300자를 고른 것은 값이 좋으면서 문장을 중간에서 끊지 않아 생성 단계에서도 근거가 읽히기 때문입니다.

---
## 마무리 1. 놓쳤던 질문이 어디까지 올라왔는지 확인하기
평균이 올랐다는 것과 **그 질문이 실제로 구제됐다**는 것은 다른 이야기입니다. 3단계에서 놓쳤던 두 질문을 개선한 색인으로 다시 검색해 **몇 등**에 올라왔는지 확인합니다.

**요구사항**:
- 3단계의 `failed_docs` 에 해당하는 평가셋 질문들을 `tuned_col` 에서 **k=3** 으로 다시 검색하세요.
- **`{정답 문서 id: 등수}`** 모양의 딕셔너리 **`rescued`** 를 만드세요. 등수는 **1부터** 셉니다.
- 결과를 출력해 두 질문이 몇 등으로 올라왔는지 보세요.

**예시**: `rescued` 는 `{'q30': ..., 'q90': ...}` 처럼 문서 id 를 키로, 1 이상 3 이하의 정수를 값으로 갖습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 놓쳤던 문서마다 그 문서를 정답으로 가진 평가셋 행을 찾아, 새 색인에서 검색하고 순서를 센다.

세부구현:
1. 빈 딕셔너리를 만들고 놓친 문서 id 를 하나씩 돈다.
2. 평가셋에서 그 id 를 정답으로 가진 행을 찾아 질문을 꺼낸다.
3. 개선한 색인에서 문서 3개를 검색한다.
4. 검색 결과에서 그 id 가 몇 번째인지 찾아(1부터) 딕셔너리에 담는다.
```

</details>

In [ ]:
rescued = {}
for doc_id in failed_docs:
    row = qna_eval[qna_eval['relevant'].apply(
        lambda r: doc_id in str(r).split('|'))].iloc[0]
    predicted = search_docs(tuned_col, row['query'], 3)
    rescued[doc_id] = predicted.index(doc_id) + 1
    print(f"{doc_id}: {rescued[doc_id]}등  {predicted} <- {row['query'][:30]}")

In [ ]:
# [자가채점]
assert set(rescued) == set(failed_docs)
assert all(1 <= rank <= 3 for rank in rescued.values()), \
    '놓쳤던 질문이 아직 상위 3 밖에 있습니다'
# 0부터 세지 않았는지 — 등수는 1부터다
assert min(rescued.values()) >= 1
# 실제 검색 결과와 맞는지 다시 계산해 대조한다
for doc_id, rank in rescued.items():
    row = qna_eval[qna_eval['relevant'].apply(
        lambda r: doc_id in str(r).split('|'))].iloc[0]
    assert search_docs(tuned_col, row['query'], 3)[rank - 1] == doc_id
print('✅ 통과!')

### 해설

`q90` 은 **1등**으로, `q30` 은 **3등**으로 올라왔습니다. 두 질문의 사정이 조금 다릅니다.

`q90`(클라우드 수탁자)은 문서가 길어 묻혀 있던 대목이 조각으로 떨어져 나오자 곧바로 1등이 됐습니다 — **진단이 맞았다**는 증거예요. `q30`(만족도 조사)은 3등에 걸쳤습니다. 상위 3 안에 들었으니 Hit 은 1이지만 MRR 로는 0.333 이라, **아직 완전히 해결된 것은 아닙니다.** 여기서 개선을 한 번 더 한다면 다음에 바꿀 설정을 찾아 같은 절차를 반복하게 됩니다.

"평균이 올랐다" 로 끝내지 않고 **개별 질문까지 되짚는 습관**이 중요합니다. 평균은 좋아졌는데 정작 중요한 질문이 여전히 실패하는 경우가 실무에서 흔합니다.

`q30` 의 상위 3은 `q48`(ARS 로 홍보 동의 받기)·`q33`(민원인 전화번호 전달)·`q30` 순입니다. "동의를 받아야 하나"라는 말이 여러 질의응답에 흩어져 있어 **어휘로는 갈리지 않는** 질문이라, 청킹을 더 잘게 해도 개선되지 않습니다. 다음 개선에서 손댈 곳은 청크 크기가 아니라 **분야 필터나 질의 다듬기**라는 뜻이고, **그 판단의 근거가 바로 이 3위라는 숫자**입니다.

---
## 마무리 2. 고친 파이프라인으로 근거 있는 답 만들기
마지막으로 개선한 색인을 실제 답변에 연결합니다. 검색이 좋아졌으니 **근거도 좋아집니다.**

**요구사항**:
- `tuned_col` 에서 `'탈퇴한 회원의 회원번호만 남겨 두어도 되나요'` 와 가까운 조각 **3개**를 가져오세요(`documents` 와 `metadatas` 를 함께 받습니다).
- 조각 본문과 그 출처(`분야`·`조항`·`쪽`)를 이어 붙여 근거를 만들고, `client.chat.completions.create(model='gpt-4o-mini', temperature=0, max_tokens=400, ...)` 로 답을 만들어 변수 **`final_answer`** 에 담으세요.
- system 메시지 규칙: **주어진 근거만 사용**하고, 답의 마지막 줄에 **`(출처: 분야 조항, N쪽)`** 형식으로 근거를 밝힐 것.
- 함께, 그 답이 쓴 근거의 문서 id 들을 리스트 **`used_docs`** 에 담으세요(검색 결과 메타데이터의 `doc_id`, 중복 없이 순서대로).

**예시**: `final_answer` 는 `(출처: 민간사업자 §21, 85쪽)` 같은 줄로 끝나는 문자열이고, `used_docs` 에는 `'q64'` 가 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 근거 하나마다 그 출처를 바로 앞줄에 적어 함께 넘긴다.

세부구현:
1. 질문을 임베딩해 조각 3개를 조회한다(본문과 메타데이터를 함께 받는다).
2. 본문과 메타데이터를 짝지어 돌며 '출처 표기 + 본문' 을 한 묶음씩 만들어 하나로 잇는다.
   2-1. 같은 반복에서 문서 id 를 중복 없이 모아 둔다.
3. 규칙을 적은 system 메시지와 근거·질문을 담은 user 메시지로 호출한다.
4. 응답에서 내용 문자열을 꺼내 변수에 담고 출력한다.
```

</details>

In [ ]:
qe = embed_model.encode(['탈퇴한 회원의 회원번호만 남겨 두어도 되나요'], normalize_embeddings=True)
res = tuned_col.query(query_embeddings=qe.tolist(), n_results=3)

parts, used_docs = [], []
for text, meta in zip(res['documents'][0], res['metadatas'][0]):
    source = f"{meta['field']} {meta['article']}, {meta['page']}쪽"
    parts.append(f'[{source}]\n{text}')
    if meta['doc_id'] not in used_docs:
        used_docs.append(meta['doc_id'])
evidence = '\n\n'.join(parts)

system = ('아래 근거만 사용해 한국어로 답하세요.\n'
          '답의 마지막 줄에 (출처: 분야 조항, N쪽) 형식으로 사용한 근거를 밝히세요.')
user = f'근거:\n{evidence}\n\n질문: 탈퇴한 회원의 회원번호만 남겨 두어도 되나요'
resp = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=400,
    messages=[{'role': 'system', 'content': system},
              {'role': 'user', 'content': user}])
final_answer = resp.choices[0].message.content
print(final_answer)
print()
print(f'쓴 근거 문서: {used_docs}')

In [ ]:
# [자가채점]
import re

assert isinstance(final_answer, str) and len(final_answer) > 30
assert '(출처:' in final_answer, '답의 마지막에 출처 표기가 없습니다'
assert 'q64' in used_docs, '탈퇴·파기 질의응답이 근거에 없습니다'
assert len(used_docs) == len(set(used_docs)), 'used_docs 에 같은 문서가 두 번 들어갔습니다'
# 답에 적힌 쪽이 실제로 검색된 조각의 쪽인지 — 지어낸 출처를 걸러 낸다
probe_vec = embed_model.encode(['탈퇴한 회원의 회원번호만 남겨 두어도 되나요'], normalize_embeddings=True)
probe = tuned_col.query(query_embeddings=probe_vec.tolist(), n_results=3)
real_pages = {mm['page'] for mm in probe['metadatas'][0]}
cited_pages = {int(n) for n in re.findall(r'(\d+)\s*쪽', final_answer)}
assert cited_pages and cited_pages <= real_pages, \
    f'검색되지 않은 쪽을 인용했습니다: {cited_pages - real_pages}'
# used_docs 가 실제 검색 결과에서 나온 것인지 대조한다
assert used_docs == list(dict.fromkeys(mm['doc_id'] for mm in probe['metadatas'][0]))
print('✅ 통과!')

### 해설

개선한 색인을 쓰면 근거가 **문서 전체가 아니라 그 대목**이라, 모델에게 넘기는 글자 수도 줄고 답도 더 또렷해집니다. 검색 품질을 올리는 일이 생성 품질로 이어지는 지점이에요.

`used_docs` 를 따로 남기는 이유는 운영 때문입니다. 사용자가 "이 답 어디서 나왔어요?" 라고 물으면 문서 id 로 되짚을 수 있어야 하고, 나중에 그 답이 틀렸다는 신고가 들어왔을 때 **어느 문서가 문제였는지** 찾아갈 수 있어야 합니다.

---
## 마무리 3. 정리해 보기 (서술형)
숫자를 냈으니 **말로 설명할 수 있어야** 내 것이 됩니다. 아래 두 물음에 각각 두세 문장으로 답하세요. 정답 노트북의 모범 서술과 비교해 보면 됩니다.

**물음 1**: 문서를 통째로 색인했을 때 `q90` 같은 **긴 문서**가 왜 검색에서 밀렸는지, 그리고 조각으로 나누니 왜 올라왔는지 설명하세요.

**물음 2**: 그렇다면 조각을 **한없이 잘게** 자르면 계속 좋아질까요? 그렇지 않다면 무엇이 나빠지는지 두 가지를 드세요.

> 아래 셀은 채점하지 않습니다. 자기 말로 적어 보세요.

### 모범 서술

**물음 1.** 문서를 통째로 임베딩하면 **벡터 하나가 그 문서 전체의 평균적인 뜻**을 나타냅니다. `q90` 은 클라우드 사업자의 지위·계약·감독·책임을 두루 다루는 1,043자짜리 문서인데, 질문은 그중 "계약을 따로 맺어야 하나" 한 대목만 묻습니다. 여러 주제가 섞이며 그 대목의 특징이 흐려져 질문 벡터와 멀어진 것입니다. 문단 단위로 나누면 그 대목이 **자기 벡터를 갖게 되어** 질문과 곧바로 맞아떨어집니다(실제로 1등으로 올라왔습니다).

**물음 2.** 계속 좋아지지 않습니다. 두 가지가 나빠집니다.

1) **문맥이 끊깁니다.** 조각이 너무 짧으면 "그렇지 않습니다" 같은 문장만 남아 무엇에 대한 답인지 알 수 없게 됩니다. 검색으로 찾아도 생성 단계에서 근거 구실을 못 합니다.

2) **찾아야 할 대상이 늘고 흩어집니다.** 조각 수가 늘어 임베딩·저장 비용이 커지고, 한 문서의 내용이 여러 조각에 흩어져 상위 K 개를 같은 문서의 조각들이 차지하기 쉽습니다. 그래서 문서 단위로 접는 과정이 더 중요해집니다.

그러므로 청크 크기는 "작을수록 좋다"가 아니라 **자료에 맞는 값을 평가셋으로 찾는 것**입니다. 이 자료에서는 문단 300자가 가장 좋았지만, 문서가 더 길고 주제가 더 섞인 자료라면 답이 달라집니다.

---
수고했어요! **기준선을 만들고 → 재고 → 실패를 진단하고 → 한 곳만 고쳐 → 다시 재는** 과정을 처음부터 끝까지 한 번 수행했습니다.

이 절차는 청킹 말고 다른 설정(임베딩 모델·K·메타데이터 필터·근거의 양)에도 그대로 씁니다. 바꾸는 것이 무엇이든 **한 번에 하나씩, 같은 방식으로** 재는 것이 전부예요.